# 19 — Diffusion Language Models

In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

Autoregressive LMs generate left-to-right. Diffusion language models iteratively denoise a corrupted sequence, often enabling non-left-to-right generation and editing-style workflows.

This notebook implements a toy discrete masking diffusion process.

In [ ]:
vocab = ['<mask>', 'ai', 'builds', 'reliable', 'products', 'with', 'evals', '.']
stoi = {w:i for i,w in enumerate(vocab)}
mask_id = stoi['<mask>']
seq = torch.tensor([[stoi[w] for w in ['ai','builds','reliable','products','.']]])

def corrupt(x, t, max_t=10):
    # higher t means more masking
    p = t / max_t
    noise = torch.rand_like(x.float()) < p
    x_t = x.clone()
    x_t[noise] = mask_id
    return x_t, noise

for t in [1,3,7,10]:
    x_t, m = corrupt(seq, t)
    print('t=', t, [vocab[i] for i in x_t[0].tolist()])


In [ ]:
class TinyDenoiser(nn.Module):
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.net = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, vocab_size))
    def forward(self, x):
        return self.net(self.emb(x))

train_sequences = torch.tensor([
    [stoi['ai'], stoi['builds'], stoi['reliable'], stoi['products'], stoi['.']],
    [stoi['ai'], stoi['builds'], stoi['products'], stoi['with'], stoi['evals']],
    [stoi['reliable'], stoi['products'], stoi['with'], stoi['evals'], stoi['.']],
])
denoiser = TinyDenoiser(len(vocab))
opt = torch.optim.AdamW(denoiser.parameters(), lr=0.03)
for step in range(120):
    clean = train_sequences[torch.randint(0, len(train_sequences), (4,))]
    t = random.randint(1, 10)
    noisy, mask = corrupt(clean, t)
    logits = denoiser(noisy)
    loss = F.cross_entropy(logits[mask], clean[mask]) if mask.any() else logits.sum()*0
    opt.zero_grad(); loss.backward(); opt.step()
print('last loss:', float(loss))


In [ ]:
@torch.no_grad()
def denoise_from_masks(length=5, steps=6):
    x = torch.full((1,length), mask_id, dtype=torch.long)
    for s in range(steps, 0, -1):
        logits = denoiser(x)
        probs = torch.softmax(logits, dim=-1)
        pred = probs.argmax(-1)
        # reveal a subset of masked positions each step
        masked = (x == mask_id).nonzero(as_tuple=False)
        if len(masked) == 0: break
        n_reveal = max(1, len(masked)//s)
        for b,pos in masked[:n_reveal]:
            x[b,pos] = pred[b,pos]
        print('step', s, [vocab[i] for i in x[0].tolist()])
    return x

denoise_from_masks()


## Product notes

Diffusion-style language models are conceptually useful for editing, infilling, and parallel refinement. For most current business products, autoregressive LMs remain the default because tooling, serving, and model availability are stronger.